# Enterprise IT Support Agent

Extends the basic ticket-management agent with three enterprise-grade capabilities:

1. **SQLite-backed history provider** (hybrid: in-memory `state["messages"]` cache + flush to `db/chat_history.db` on save).
2. **Three class-based, agent-level middleware**: input guardrail, exception handling, logging.
3. **Session serialize / resume demo** at the end to prove SQLite-backed continuity.

All ticket tools are reused unchanged from `lib/ticket_management/tools.py`.

In [ ]:
# Cell 1 — Imports, environment, sys.path
import os
import re
import sys
import json
import time
import logging
from collections.abc import Awaitable, Callable, Sequence
from pathlib import Path
from typing import Any

from azure.identity.aio import AzureCliCredential
from agent_framework import (
    AgentContext,
    AgentMiddleware,
    AgentResponse,
    AgentSession,
    HistoryProvider,
    Message,
    MiddlewareTermination,
)
from agent_framework.foundry import FoundryChatClient
from dotenv import load_dotenv
from sqlalchemy.exc import SQLAlchemyError

# Load .env from the day2-usecase root (one level up from notebooks/)
load_dotenv(override=True)

# Make the lib/ package importable
LIB_PATH = str(Path("./lib").resolve())
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from ticket_management.database import init_db
from ticket_management.chat_history_models import (
    BlockedQuery,
    ChatMessage,
    chat_engine,
    get_chat_session,
    init_chat_history_db,
)
from ticket_management.tools import (
    close_ticket,
    get_tickets_by_registered_by,
    get_tickets_by_resolved_by,
    register_ticket,
    resolve_ticket,
    search_tickets,
)

print("Imports loaded.")

In [ ]:
# Cell 2 — Initialise both databases
init_db()
init_chat_history_db()

# Ensure logs directory exists
LOGS_DIR = Path("../logs").resolve()
LOGS_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOGS_DIR / "agent.log"

BLACKLIST_PATH = Path("../db/blacklist.txt").resolve()

print(f"Tickets DB ready.")
print(f"Chat-history DB ready.")
print(f"Log file: {LOG_FILE}")
print(f"Blacklist file: {BLACKLIST_PATH}")

## SQLite-backed History Provider

Hybrid strategy:
- `get_messages` returns `state["messages"]` if already cached; otherwise loads rows from SQLite, hydrates them into `Message` objects, caches them, and returns.
- `save_messages` appends to `state["messages"]` AND inserts new rows into SQLite (`chat_messages` table).

This means the in-memory cache stays hot during a session, and SQLite acts as the durable store for resume/rehydration.

In [ ]:
# Cell 3 — SqliteHistoryProvider
class SqliteHistoryProvider(HistoryProvider):
    """History provider that caches in state['messages'] and flushes to SQLite on save."""

    def __init__(self) -> None:
        super().__init__("sqlite-history")

    async def get_messages(
        self,
        session_id: str | None,
        *,
        state: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> list[Message]:
        if state is None:
            return []

        # Hot cache hit
        cached = state.get("messages")
        if cached:
            return list(cached)

        # Cold start: hydrate from SQLite
        if not session_id:
            return []

        with get_chat_session() as db:
            rows = (
                db.query(ChatMessage)
                .filter(ChatMessage.session_id == session_id)
                .order_by(ChatMessage.id.asc())
                .all()
            )
            hydrated: list[Message] = []
            for r in rows:
                # content stored as JSON-serialised text; fall back to plain text
                try:
                    content_obj = json.loads(r.content)
                except (ValueError, TypeError):
                    content_obj = r.content
                hydrated.append(Message(role=r.role, contents=[content_obj] if not isinstance(content_obj, list) else content_obj))

        state["messages"] = hydrated
        return list(hydrated)

    async def save_messages(
        self,
        session_id: str | None,
        messages: Sequence[Message],
        *,
        state: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> None:
        if state is None:
            return

        # Update in-memory cache
        existing = state.get("messages", [])
        state["messages"] = [*existing, *messages]

        # Flush new messages to SQLite
        if not session_id:
            return

        with get_chat_session() as db:
            for m in messages:
                # serialise content list to JSON string (best-effort)
                try:
                    content_text = json.dumps([
                        c if isinstance(c, (str, int, float, bool)) else getattr(c, "text", str(c))
                        for c in (m.contents or [])
                    ])
                except (TypeError, ValueError):
                    content_text = str(m.contents)
                db.add(ChatMessage(session_id=session_id, role=str(m.role), content=content_text))

print("SqliteHistoryProvider defined.")

## Middleware (all class-based, agent-level)

Order on the agent: `[InputGuardrailMiddleware, ExceptionHandlingAgentMiddleware, LoggingAgentMiddleware]`.
Onion model — guardrail is outermost (cheapest reject); logging is innermost so it captures everything else.

In [ ]:
# Cell 4 — LoggingAgentMiddleware
_LOGGER_NAME = "enterprise_ticket_agent"

def _configure_logger() -> logging.Logger:
    logger = logging.getLogger(_LOGGER_NAME)
    if logger.handlers:
        return logger  # already configured
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    logger.propagate = False
    return logger


class LoggingAgentMiddleware(AgentMiddleware):
    """Logs each agent run: inbound query, message count, duration, response snippet."""

    def __init__(self) -> None:
        self.logger = _configure_logger()

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        last = context.messages[-1] if context.messages else None
        last_text = (last.text if last and last.text else "").replace("\n", " ")[:200]
        self.logger.info(
            f"[run-start] msgs={len(context.messages or [])} last_user={last_text!r}"
        )

        start = time.perf_counter()
        try:
            await call_next()
        finally:
            duration = time.perf_counter() - start
            response_snippet = ""
            if context.result is not None:
                try:
                    response_snippet = str(getattr(context.result, "text", context.result))[:200]
                except Exception:
                    response_snippet = "<unserializable>"
            self.logger.info(
                f"[run-end] duration={duration:.3f}s response={response_snippet!r}"
            )

print("LoggingAgentMiddleware defined.")

In [ ]:
# Cell 5 — ExceptionHandlingAgentMiddleware
class ExceptionHandlingAgentMiddleware(AgentMiddleware):
    """Catches exceptions during the run and converts them into a friendly response.

    Handles TimeoutError, ValueError, SQLAlchemyError, then any other Exception.
    On error, sets context.result and returns (does NOT re-raise — graceful degradation).
    """

    def __init__(self) -> None:
        self.logger = _configure_logger()

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        try:
            await call_next()
        except TimeoutError as e:
            self.logger.error(f"[exception] TimeoutError: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    "The request timed out while contacting a downstream service. "
                    "Please try again in a moment."
                ])]
            )
        except ValueError as e:
            self.logger.error(f"[exception] ValueError: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    f"I couldn't process that request because of invalid input: {e}"
                ])]
            )
        except SQLAlchemyError as e:
            self.logger.error(f"[exception] SQLAlchemyError: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    "A database error occurred while processing your request. "
                    "Our team has been notified — please try again shortly."
                ])]
            )
        except Exception as e:  # noqa: BLE001 — catch-all is the point
            self.logger.exception(f"[exception] Unhandled: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    "Sorry, something went wrong while handling your request. "
                    "Please try again or contact support."
                ])]
            )

print("ExceptionHandlingAgentMiddleware defined.")

In [ ]:
# Cell 6 — InputGuardrailMiddleware
class InputGuardrailMiddleware(AgentMiddleware):
    """Blocks queries containing blacklisted words (whole-word, case-insensitive).

    On match: logs the attempt, persists a row into `blocked_queries`, sets a polite
    refusal response (citing the matched word as the reason), and raises
    MiddlewareTermination to short-circuit the run.
    """

    def __init__(self, blacklist_path: Path) -> None:
        self.logger = _configure_logger()
        self.words = self._load_words(blacklist_path)
        if self.words:
            pattern = r"\b(" + "|".join(re.escape(w) for w in self.words) + r")\b"
            self.regex = re.compile(pattern, re.IGNORECASE)
        else:
            self.regex = None

    @staticmethod
    def _load_words(path: Path) -> list[str]:
        if not path.exists():
            return []
        out: list[str] = []
        for line in path.read_text(encoding="utf-8").splitlines():
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            out.append(stripped.lower())
        return out

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        if self.regex is None:
            await call_next()
            return

        last = context.messages[-1] if context.messages else None
        text = last.text if last and last.text else ""
        m = self.regex.search(text)

        if m:
            matched = m.group(1)
            session_id = getattr(context.session, "id", None) if context.session else None
            self.logger.warning(f"[guardrail] BLOCKED matched={matched!r} session={session_id}")

            # Persist blocked attempt
            try:
                with get_chat_session() as db:
                    db.add(BlockedQuery(
                        session_id=session_id,
                        query=text,
                        matched_word=matched,
                    ))
            except Exception as e:  # noqa: BLE001
                self.logger.error(f"[guardrail] failed to persist blocked query: {e}")

            refusal = (
                f"I'm sorry, I can't help with that request. Your message contained the word "
                f"'{matched}', which is on our blocked list (offensive / inappropriate / unsafe "
                "content). Please rephrase your question without that term and I'll be happy to assist."
            )
            context.result = AgentResponse(messages=[Message("assistant", [refusal])])
            raise MiddlewareTermination(result=context.result)

        await call_next()

print("InputGuardrailMiddleware defined.")

## Build the agent

Tools, history provider, and middleware are wired in here. Middleware execution order: guardrail → exception handling → logging → agent → logging → exception handling → guardrail (onion model).

In [ ]:
# Cell 7 — Build the FoundryChatClient and the agent
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=credential,
)

history_provider = SqliteHistoryProvider()

agent = client.as_agent(
    name="ITSupportAgent",
    instructions=(
        "You are a helpful IT support management agent for a company called DataTech.\n"
        "You assist both end users who want to raise IT support tickets and IT admin staff who resolve or close them.\n"
        "\n"
        "Guidelines:\n"
        "- Always use the provided tools — never fabricate ticket IDs or statuses.\n"
        "- When registering a ticket, confirm the ticket ID returned by the tool.\n"
        "- When resolving a ticket, ensure you capture who is resolving it and the resolution details.\n"
        "- When closing a ticket, confirm the final status from the tool response.\n"
        "- Address the user by name whenever it is known from the conversation.\n"
        "- Valid priorities: Low, Medium, High, Critical.\n"
        "- Valid statuses: Open, In Progress, On Hold, Resolved, Closed, Cancelled."
    ),
    tools=[
        register_ticket,
        get_tickets_by_registered_by,
        get_tickets_by_resolved_by,
        search_tickets,
        resolve_ticket,
        close_ticket,
    ],
    context_providers=[history_provider],
    middleware=[
        InputGuardrailMiddleware(BLACKLIST_PATH),
        ExceptionHandlingAgentMiddleware(),
        LoggingAgentMiddleware(),
    ],
)

print("ITSupportAgent (enterprise) ready.")

In [ ]:
# Cell 8 — Shared session for all multi-turn scenarios
session = agent.create_session()
session_id = getattr(session, "session_id", None)
print(f"Session created. id={session_id}")

## Scenarios 1–4 — Existing ticket flow (reused)

In [ ]:
# Cell 9 — Scenario 1: Jessie raises a ticket
print("=" * 60)
print("SCENARIO 1: Jessie raises a ticket")
print("=" * 60)

result = await agent.run(
    "Hi, I'm Jessie Thompson. My laptop has completely stopped connecting to the office VPN since this morning. "
    "I've tried restarting multiple times but nothing works. This is blocking me from accessing all internal systems. "
    "Can you please raise a High priority support ticket for me?",
    session=session,
)
print(f"Agent: {result}\n")

In [11]:
# Cell 10 — Scenario 1 follow-up: Jessie asks for confirmation
result = await agent.run(
    "Thanks! Can you confirm the ticket ID and the current status for me?",
    session=session,
)
print(f"Agent: {result}\n")

Agent: Certainly, Jessie! Here are the ticket details:

- **Ticket ID:** DTKT10001  
- **Current Status:** Open  

Let me know if you need further assistance!



In [12]:
# Cell 11 — Scenario 2: IT Admin resolves Jessie's ticket
print("=" * 60)
print("SCENARIO 2: IT Admin resolves the ticket")
print("=" * 60)

result = await agent.run(
    "This is the IT Admin team. I'm Aarav Sharma. "
    "We've investigated Jessie Thompson's VPN connectivity issue. "
    "Root cause: the VPN client certificate had expired and was not auto-renewed due to a misconfigured Group Policy. "
    "We've reissued the certificate, pushed the updated GPO, and verified that Jessie's laptop can now connect to the VPN successfully. "
    "Please mark Jessie's open VPN ticket as resolved with these details.",
    session=session,
)
print(f"Agent: {result}\n")

SCENARIO 2: IT Admin resolves the ticket
Agent: The ticket has been successfully marked as resolved. Here are the updated details:

- **Ticket ID:** DTKT10001  
- **Description:** Laptop not connecting to office VPN.  
- **Priority:** High  
- **Status:** Resolved  
- **Resolved By:** Aarav Sharma  
- **Resolution Remarks:** The VPN client certificate had expired and was not auto-renewed due to a misconfigured Group Policy. Reissued the certificate, pushed the updated GPO, and verified successful VPN connectivity on Jessie's laptop.  
- **Resolved Date:** 2026-05-08  

Let me know if there’s anything else you need!



In [13]:
# Cell 12 — Scenario 3: IT Admin closes the resolved ticket
print("=" * 60)
print("SCENARIO 3: IT Admin closes the ticket")
print("=" * 60)

result = await agent.run(
    "IT Admin here again — Aarav Sharma. "
    "Jessie has confirmed that the VPN is working perfectly now. "
    "Please go ahead and close that resolved VPN ticket.",
    session=session,
)
print(f"Agent: {result}\n")

SCENARIO 3: IT Admin closes the ticket
Agent: The ticket has been successfully closed. Here are the final details:

- **Ticket ID:** DTKT10001  
- **Description:** Laptop not connecting to office VPN.  
- **Priority:** High  
- **Status:** Closed  
- **Resolved By:** Aarav Sharma  
- **Resolution Remarks:** The VPN client certificate had expired and was not auto-renewed due to a misconfigured Group Policy. Reissued the certificate, pushed the updated GPO, and verified successful VPN connectivity on Jessie's laptop.  
- **Resolved Date:** 2026-05-08  

Let me know if there’s anything else you need assistance with!



In [14]:
# Cell 13 — Scenario 4: List all open tickets
print("=" * 60)
print("SCENARIO 4: IT Admin queries all open tickets")
print("=" * 60)

result = await agent.run(
    "Before we wrap up, can you provide me with a list of all currently open tickets in the system?",
    session=session,
)
print(f"Agent: {result}\n")

SCENARIO 4: IT Admin queries all open tickets
Agent: There are currently no open tickets in the system. Let me know if there's anything else you'd like to check or resolve!



## Scenario 5 — Input Guardrail demo

The query below contains a word from `db/blacklist.txt` (e.g., `idiot`). The guardrail middleware should match it, log + persist a `blocked_queries` row, and short-circuit the run with a polite refusal.

In [15]:
# Cell 14 — Scenario 5: Guardrail blocks an offensive query
print("=" * 60)
print("SCENARIO 5: Guardrail blocks an offensive query")
print("=" * 60)

offensive_query = "This is so frustrating, the IT admin is an idiot — please raise a ticket about it."
result = await agent.run(offensive_query, session=session)
print(f"Agent: {result}\n")

# Verify a row was logged in blocked_queries
with get_chat_session() as db:
    blocked_count = db.query(BlockedQuery).filter(BlockedQuery.session_id == session_id).count()
print(f"blocked_queries rows for this session: {blocked_count}")

SCENARIO 5: Guardrail blocks an offensive query
Agent: I'm here to assist with IT-related issues in a constructive and professional manner. If you have specific concerns or feedback about IT support, I recommend clearly outlining the issue so it can be addressed effectively. Would you like to rephrase your request or describe the situation in more detail so I can help you?

blocked_queries rows for this session: 0


## Scenario 6 — Exception handling demo

We deliberately invoke the underlying repository with a value that triggers an internal error (e.g., calling `resolve_ticket` directly with an unknown ID). The `ExceptionHandlingAgentMiddleware` should intercept the failure and return a friendly message instead of crashing.

_Note: the existing tools wrap their own exceptions and return error strings, so to actually exercise the agent-level exception middleware we ask the agent to perform an action that causes an internal error during the run._

In [16]:
# Cell 15 — Scenario 6: Exception middleware degrades gracefully
print("=" * 60)
print("SCENARIO 6: Exception middleware demo")
print("=" * 60)

# Force a controlled error by monkey-patching one tool to raise on this turn only.
import ticket_management.repository as _repo
_orig_resolve = _repo.resolve_ticket

def _broken_resolve(*args, **kwargs):
    raise TimeoutError("Simulated downstream timeout while resolving ticket")

_repo.resolve_ticket = _broken_resolve
try:
    result = await agent.run(
        "Please resolve ticket TCK-9999 with note 'force-resolve test'.",
        session=session,
    )
    print(f"Agent: {result}\n")
finally:
    _repo.resolve_ticket = _orig_resolve
    print("(repository.resolve_ticket restored)")

SCENARIO 6: Exception middleware demo
Agent: It seems there was an issue resolving ticket TCK-9999 due to a simulated timeout from the system. You may want to try again after some time or check the status of the ticket to ensure it is still active. Let me know how you'd like to proceed!

(repository.resolve_ticket restored)
